# 语音识别（在机械臂主机上运行）

把项目里的 `src/voice`（A 侧 E2 语音模块）搬到机械臂主机：麦克风 → Vosk 离线中文识别 →
关键词“帮我分类”命中 → 输出 `VoiceCommand`（`voice_code=1`，SORT_ONE）。

## 一、上传文件

```
project/
├── voice/
│   ├── app.py / recognizer.py / command_gate.py / voice_command.py
│   └── models/vosk-cn/     # 中文模型（几百 MB，必须完整上传）
├── voice_live.ipynb
```

## 二、系统依赖（Linux 板子必做，只需一次）

```bash
sudo apt-get update
sudo apt-get install -y portaudio19-dev python3-pyaudio
```

然后依次运行下面的单元：安装 Python 包 → 定位模型 → 麦克风自检 → 实时识别 → 无麦克风校验。

In [ ]:
%pip install -q vosk PyAudio

In [ ]:
from pathlib import Path
import sys, os, subprocess, time, json

# 自动定位上传后的工程根目录（包含 gesture/ 和 voice/ 的那一层）
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_SRC = next((p for p in candidates if (p / "voice").is_dir()), None)
if PROJECT_SRC is None:
    raise SystemExit("未找到 voice 目录：请先把 src/voice 上传到当前目录下")

sys.path.insert(0, str(PROJECT_SRC))
print("PROJECT_SRC =", PROJECT_SRC)

from voice.voice_command import VOICE_CODES
from voice.recognizer import DEFAULT_GRAMMAR

model_dir = PROJECT_SRC / "voice" / "models" / "vosk-cn"
need = ["am", "conf", "graph", "ivector"]
missing = [n for n in need if not (model_dir / n).exists()]
print("模型目录:", model_dir)
print("缺失部分:", missing or "无")
print("命令映射:", VOICE_CODES)
print("受限语法:", DEFAULT_GRAMMAR)
assert not missing, "Vosk 模型不完整，请重新上传 voice/models/vosk-cn 整个目录"

In [ ]:
# 麦克风自检：列出输入设备，确认能拿到音频
import pyaudio
pa = pyaudio.PyAudio()
info = pa.get_host_api_info_by_index(0)
print("默认输入设备索引:", info.get("defaultInputDevice"))
for i in range(pa.get_device_count()):
    d = pa.get_device_info_by_index(i)
    if d.get("maxInputChannels", 0) > 0:
        print(f"[{i}] {d['name']}  channels={d['maxInputChannels']}  rate={int(d['defaultSampleRate'])}")
pa.terminate()

In [ ]:
# 实时语音识别：跑 30 秒，识别到的文本与命中的命令都会打印
import subprocess, time, json
from pathlib import Path

cmd = [sys.executable, "-m", "voice.app", "--mic", "--asr", "vosk",
       "--vosk-model", str(PROJECT_SRC / "voice" / "models" / "vosk-cn")]
print(" ".join(cmd))

proc = subprocess.Popen(cmd, cwd=str(PROJECT_SRC), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
deadline = time.time() + 30
out = []
try:
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            break
        line = line.rstrip()
        out.append(line)
        if line:
            print(line, flush=True)
finally:
    proc.kill()
    proc.wait(timeout=5)

cmds = [json.loads(x) for x in out if x.startswith("{")]
print(f"命中命令 {len(cmds)} 条")
for c in cmds:
    print(c.get("text"), "voice_code=", c.get("voice_code"), "task_id=", c.get("task_id"))

In [ ]:
# 无麦克风时的链路验证（直接喂文本，验证关键词匹配与冷却逻辑）
from voice.recognizer import MockTextBackend, VoiceRecognizer
from voice.command_gate import VoiceCommandGate
from voice.voice_command import VoiceCommand, VOICE_CODES
import time

items = [("帮我分类", 0.95), ("今天天气不错", 0.9), ("帮我分类", 0.95)]
rec = VoiceRecognizer(MockTextBackend(items), min_confidence=0.5)
gate = VoiceCommandGate(cooldown_s=3.0, min_confidence=0.5)
now = time.time()
for utt in rec.backend:
    name = VoiceRecognizer.match_command(utt.text)
    print("识别:", utt.text, "conf=", utt.confidence, "命中:", name)
    if name is not None:
        cmd = VoiceCommand(text=utt.text, confidence=utt.confidence,
                           voice_code=VOICE_CODES[name])
        emitted = gate.update(cmd, now)
        now += 1.0
        print("  下发:", emitted.to_json() if emitted else "被冷却/门限拦截")

## 三、结果怎么用

命中的命令会打印成 `VoiceCommand` JSON：

```json
{"text":"帮我分类","confidence":0.93,"command_id":"vc_xxx","voice_code":1,"source":"voice","captured_at":"2026-11-06T16:20:00","latency_ms":0.0,"task_id":null}
```

C 侧读取 `voice_code == 1`（SORT_ONE）后执行一次完整分拣，并在 `task_id` 上补全 B 的编排编号。
真实命令由麦克风识别后直接写出一行 JSON；不接后端时，保存到文件或复制给 C 即可。

## 四、常见问题

- `No module named vosk` → 先跑安装单元，注意板子需 64 位系统。
- 能听到声音但一直“无关语音” → 检查是否说了“帮我分类”，或把 `--no-grammar` 打开做自由解码调试。
- 反复触发同一命令 → 冷却默认 3 秒，可调 `--cooldown-s`。